# Complete decision analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xmaster6y/lczerolens/blob/main/docs/source/notebooks/tutorials/decision-analysis.ipynb)

This notebook runs the maintained end-to-end workflow: evaluator preference, replayable search, exact line evidence, a constrained counterfactual, an authored puzzle, and canonical serialization. The deterministic fixture makes the composition reproducible; its choices are not scientific conclusions.

In [ ]:
# Colab starts from a clean runtime; local and docs builds skip this setup.
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

if importlib.util.find_spec("google.colab") is not None:
    checkout = Path("/content/lczerolens")
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Xmaster6y/lczerolens.git", str(checkout)],
            check=True,
        )
    os.chdir(checkout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from examples.decision_analysis_tutorial import (
    TUTORIAL_DECISION_DIGEST,
    run_tutorial,
)

with TemporaryDirectory() as directory:
    artifact = Path(directory) / "decision.json"
    result = run_tutorial(artifact)
    artifact_size = artifact.stat().st_size

assert result.restored_decision == result.decision
assert result.decision_digest == TUTORIAL_DECISION_DIGEST
{
    "policy_move": result.decision.policy_move,
    "search_move": result.decision.search_move,
    "changed": result.decision.changed,
    "digest": result.decision_digest,
    "artifact_bytes": artifact_size,
}

The joined action table keeps policy and search observations distinct. Exact line analysis is attached only to the candidates requested by the workflow; absent line evidence is not silently invented.

In [ ]:
candidate_rows = []
for move in sorted({result.decision.policy_move, result.decision.search_move}):
    action = result.decision.actions[move]
    candidate_rows.append(
        {
            "move": move,
            "policy_rank": action.policy_rank,
            "search_visits": action.search_visits,
            "has_exact_line": action.line is not None,
        }
    )
candidate_rows

Puzzle correctness is source-authored and independent of evaluator or search preference. Counterfactual validity is likewise explicit: this workflow compares two legal sibling moves rather than claiming an arbitrary board edit is on-manifold.

In [ ]:
{
    "puzzle_status": result.puzzle_attempt.status.value,
    "counterfactual_validity": result.counterfactual.validity.value,
    "factual_move": result.counterfactual.operator.factual_move.uci(),
    "alternative_move": result.counterfactual.operator.alternative_move.uci(),
}